# Base Place Recognition Pipeline 

Test Place Recognition on the 3DSSG dataset using `opr.pipelines`

In [1]:
import itertools
import shutil
from pathlib import Path
import json

import faiss
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import plotly.graph_objects as go

from torchvision import transforms as T
from opr.datasets.itlp import ITLPCampus
#from opr.models.place_recognition import MinkLoc3D
from mmpr.inference import PlaceRecognitionPipeline, FaissFlatIndex, SequencePlaceRecognitionPipeline, PlaceRecognitionRerankPipeline

from gsloc.inference.pr_infer import PRInferencer, PRRerankInferencer
from gsloc.models import opr_graph_extention as network
# from opr.pipelines.place_recognition import PlaceRecognitionPipeline

from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-05-07 13:46:34.700 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


## Create dataset object

In [2]:
dataset_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan"
test_dir = Path("/home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan")
index_path = test_dir / "index"
query_cache_path = test_dir / "query_cache"
bench_report_dir = test_dir / "seq_benchmark_report"
graph_dir = "SceneGraphs_Makarov_FULL_TEST_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"

In [3]:
from torchvision.transforms import functional as F

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True)
])

In [4]:
from torchvision.transforms import functional as F

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.44420420130352495, 0.41322746532289134, 0.3678658064565412], std=[0.24352604373543688, 0.24045797651069503, 0.24250136992133814]),
    T.Resize([322, 322], antialias=True)
])

In [6]:
three_rscan_ds = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=index_path,
    rebuild_meta=False,  # meta.parquet already built
    # limit=20000,
    image_transform=image_transform_fn,
    save_meta=False,
    scene_filter_mode="listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
    graph_feat_dim = 4,
    graph_edge_attr_dim = 10,
    graph_rotate = True,
    graph_path=graph_dir,
    edge_normalizer_path=edge_normalizer_path,
)
# You can create your own dataloader for index generation
# dataloader = DataLoader(
#     three_rscan_ds, batch_size=16, shuffle=False, num_workers=4, collate_fn=three_rscan_ds.collate_fn
# )

## Create model

In [7]:
# model = MegaLoc()
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()

In [8]:
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OPR_GAT_graph_encoder = network.OPR_GATGraphEncoder(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=529, 
    node_emb_dim=128,
    num_edge_classes=41,
    edge_emb_dim=128,
    proj_dim=256,
    edge_cont_dim=10,
    dropout=0.1,
    heads=4
    ).to(device)
    
megaloc = torch.hub.load("gmberton/MegaLoc", "get_trained_model")
image_encoder = megaloc.to(device)

model = network.OPR_MultiModalVPRGraphEncoder(
    graph_encoder=OPR_GAT_graph_encoder,
    image_encoder=None,
    image_out_dim=8448,
    graph_out_dim=256,
    fusion_dim=8448,
    normalize=True,
    graph_fusion_scale=0.05,
    freeze_image_encoder=True,
    mode="graph")

missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

model.to(device)
model.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


OPR_MultiModalVPRGraphEncoder(
  (graph_encoder): OPR_GATGraphEncoder(
    (edge_cont_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (edge_lbl_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (node_emb): Embedding(529, 128)
    (edge_emb): Embedding(41, 128)
    (edge_cont_mlp): Sequential(
      (0): Linear(in_features=10, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_gate): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): Sigmoid()
    )
    (edge_label_proj): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_fuse): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (

In [9]:
# # Load graph+MegaLoc head from ``gatv1/best_model.pth`` (hub MegaLoc uses different prefixes; conv stack is GAT in ckpt vs GINE in ``VPRGraphEncoder``).
# weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/2026-04-17_16-35-44/best_model.pth")
# ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)

# graph_enc = network.OPR_VPRGraphEncoder(
#     in_dim=4,
#     hidden_dim=512,
#     n_layers=1,
#     num_node_classes=528 + 1,
#     num_edge_classes=41,
#     node_emb_dim=128,
#     edge_emb_dim=128,
#     proj_dim=256,
# )
# model = network.OPR_MultiModalVPRGraphEncoder(
#     graph_encoder=graph_enc,
#     image_encoder=MegaLoc().model,
#     image_out_dim=8448,
#     graph_out_dim=256,
#     fusion_dim=8448,
#     normalize=True,
#     graph_fusion_scale=0.05,
#     freeze_image_encoder=True,
#     mode="graph",
# )
# missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
# ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
# unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
# if unexpected_other:
#     raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# # ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()

## Create Index (files that are used to do retrievel based on database)

In [10]:
# generate function runs model for all dataset's elements and generates 3 files that are need for retrievel
index = FaissFlatIndex.generate(
    directory=index_path,
    dataset=three_rscan_ds,
    dataloader=None,
    model=model,
    rebuild_meta=False,
    rebuild_descriptors=False,
    batch_size = 24,
    num_workers = 6,
    shuffle = False,
    metric = "l2", # can be also "ip" - inner product
    version = 1)
print(f"Index created at {index_path}")
print(f"Index size: {index.size()}, dim: {index.dim()} metric: {index.metric()}")

2026-05-07 13:47:46.962 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
100%|██████████| 394/394 [01:06<00:00,  5.96it/s]
2026-05-07 13:48:53.059 | INFO     | mmpr.inference.index:generate:466 - descriptors.npy file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index
2026-05-07 13:48:53.064 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index


Index created at /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index
Index size: 9449, dim: 256 metric: l2


# Test PlaceRecognitionPipeline

In [11]:
pipeline = PlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
)


seq_pr_pipeline = SequencePlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
    max_window=25,
    per_frame_k=20,
    final_k=50,
    descriptor_agg="mean",
)

In [13]:
three_rscan_q = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=query_cache_path,
    # save_meta=True,
    rebuild_meta=False,
    # limit=10000,
    image_transform=image_transform_fn,
    scene_filter_mode="same_room_excluding_listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
    graph_feat_dim = 4,
    graph_edge_attr_dim = 10,
    graph_rotate = True,
    graph_path=graph_dir,
    edge_normalizer_path=edge_normalizer_path,
)

In [14]:
inferencer = PRInferencer(
    pr_pipeline=pipeline,
    query_dataset=three_rscan_q,
    batch_size=16,
    num_workers=4,
    query_cache_dir=query_cache_path,
    k=100,
    device="cuda"
)

In [15]:
# frames = inferencer.run(rebuild_query_descriptors=True)
# inferencer.save(query_cache_path / "frames.npz", frames=frames)
frames = inferencer.load(query_cache_path / "frames.npz")

In [16]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

21013 21013


100%|██████████| 21013/21013 [00:10<00:00, 2097.46it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,5437,0.258745,25.874459
1,5,21013,10500,0.499691,49.969067
2,10,21013,13162,0.626374,62.637415
3,25,21013,16598,0.789892,78.989197


In [14]:
curr_report_dir = bench_report_dir / "room_k25_batch-std_makarov"
# inferencer.save(curr_report_dir / "frames.npz", frames=itog_frames)

result_df = inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    per_frame_k_used=25,
    save_dir=curr_report_dir,
    std_mode="global",
    scene_df_field="scene",
)

  0%|          | 0/11 [00:00<?, ?it/s]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


  9%|▉         | 1/11 [01:28<14:47, 88.80s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 18%|█▊        | 2/11 [04:05<19:19, 128.80s/it]

...

 82%|████████▏ | 9/11 [41:18<10:50, 325.46s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 91%|█████████ | 10/11 [47:01<05:30, 330.73s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


100%|██████████| 11/11 [52:45<00:00, 287.79s/it]


In [ ]:
bench_report_dir = Path("/home/kartashov_ga/projects/GSLoc/data/tests/26-04-25/Fusion/Score-func/3rscan") / "seq_benchmark_report"
curr_report_dir = bench_report_dir / "room_k25_batch-std_1g-1000i"

result_df = inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    per_frame_k_used=25,
    save_dir=curr_report_dir,
    std_mode="global",
    scene_df_field="scene",
)

In [15]:
result_df

,w,auc_pr,f1_max,recall_at_1,recall_at_1_std,recall_at_5,recall_at_5_std,recall_at_10,recall_at_10_std,recall_at_25,recall_at_25_std,num_valid,num_total
0,1,0.170507,0.286933,0.258745,0.041788,0.499643,0.050845,0.626327,0.049780,0.789892,0.042026,21013,21013
1,2,0.164904,0.274806,0.269833,0.044078,0.521249,0.050726,0.651168,0.044654,0.805834,0.040358,21013,21013
2,3,0.161355,0.267483,0.273259,0.046155,0.532813,0.049461,0.662542,0.044601,0.816209,0.037824,21013,21013
3,5,0.158020,0.263721,0.276971,0.041779,0.542902,0.050557,0.676486,0.046057,0.828392,0.036707,21013,21013
4,7,0.158201,0.270559,0.278827,0.044376,0.548518,0.054615,0.684529,0.048622,0.836387,0.038584,21013,21013
5,10,0.158720,0.276678,0.281445,0.043831,0.550659,0.049885,0.690049,0.046835,0.841622,0.038199,21013,21013
6,15,0.159335,0.282926,0.279256,0.042808,0.548756,0.046517,0.690430,0.042492,0.848475,0.036897,21013,21013
7,20,0.159845,0.287783,0.276210,0.041938,0.544758,0.050646,0.687194,0.049598,0.852805,0.032601,21013,21013
8,25,0.160394,0.291713,0.272260,0.046101,0.541141,0.051385,0.679199,0.049895,0.849855,0.033942,21013,21013
9,30,0.160811,0.295061,0.271165,0.041261,0.541617,0.050454,0.672726,0.050753,0.847951,0.036122,21013,21013


#Graph random report

### Base MakarovgraphResults

In [17]:
plot_metrics_vs_window_with_stats(result_df, result_df)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('ybDbbC7TxT9LlnyulRvFP8g6B3dEp8' ... 'POh8Q/+UAmpnOVxD/Yk6bjhJXEPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [1],
               'y': [0.17050724330272818]},
              {'line': {'color': 'purple', 'dash': 'dot'},
           

In [12]:
model1 = model
model2 = MegaLoc()
model2.to(device)
model2.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main


MegaLoc(
  (model): MegaLocModel(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, 

In [12]:
index1 = index
index_path2 = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index2"
index2 = FaissFlatIndex.generate(
    directory=index_path2,
    dataset=three_rscan_ds,
    dataloader=None,
    model=model2,
    rebuild_meta=False,
    rebuild_descriptors=False,
    batch_size = 24,
    num_workers = 6,
    shuffle = False,
    metric = "l2", # can be also "ip" - inner product
    version = 1)
print(f"Index created at {index_path2}")
print(f"Index size: {index2.size()}, dim: {index2.dim()} metric: {index2.metric()}")

2026-05-05 09:18:42.195 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-05-05 09:18:42.195 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-05-05 09:18:42.255 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index2


Index created at /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index2
Index size: 9449, dim: 8448 metric: l2


In [13]:
rerank_pipeline = PlaceRecognitionRerankPipeline(
    index1=index1,
    index2=index2,
    model1=model1,
    model2=model2,
    device="cuda",
)

In [14]:
query_cache_path = test_dir / "query_cache_rerank"
rerank_inferencer = PRRerankInferencer(
    pr_rerank_pipeline=rerank_pipeline,
    query_dataset=three_rscan_q,
    batch_size=16,
    num_workers=4,
    query_cache_dir=query_cache_path,
    k=250,
    device="cuda"
)

In [27]:
frames250 = rerank_inferencer.run(rebuild_query_descriptors=True)
rerank_inferencer.save(query_cache_path / "frames250.npz", frames=frames250)

Compute descriptors + PR cache:  30%|███       | 399/1314 [02:47<05:34,  2.74it/s]2026-05-05 08:49:02.959 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:385 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000017.pt
2026-05-05 08:49:02.978 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:385 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000018.pt
Compute descriptors + PR cache: 100%|██████████| 1314/1314 [08:47<00:00,  2.49it/s]
2026-05-05 08:55:03.943 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/query_cache_rerank/meta.parquet


In [15]:
frames250 = rerank_inferencer.load(query_cache_path / "frames250.npz")

In [16]:
rerank_inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

  0%|          | 0/21013 [00:00<?, ?it/s]

100%|██████████| 21013/21013 [00:04<00:00, 4913.25it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,16645,0.792129,79.212868
1,5,21013,18237,0.867891,86.789131
2,10,21013,18811,0.895208,89.520773
3,25,21013,19595,0.932518,93.251797


In [16]:
bench_report_dir = Path("/home/kartashov_ga/projects/GSLoc/data/tests/26-04-25/Fusion/Score-func/3rscan") / "seq_benchmark_report"
curr_report_dir = bench_report_dir / "room_k25_batch-std_rerank_makarov"

result_df = rerank_inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    per_frame_k_used=25,
    final_k=25,
    save_dir=curr_report_dir,
    std_mode="global",
    scene_df_field="scene",
)

  0%|          | 0/11 [00:00<?, ?it/s]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


  9%|▉         | 1/11 [00:25<04:19, 25.96s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 18%|█▊        | 2/11 [00:51<03:53, 25.97s/it]

...

recall@k calculation started
micro curves calculation started
fused rankings preparation started
rankings creation started
ranking iteration started


 91%|█████████ | 10/11 [04:23<00:26, 26.79s/it]

recall@k calculation started
micro curves calculation started
fused rankings preparation started
rankings creation started
ranking iteration started


100%|██████████| 11/11 [04:51<00:00, 26.47s/it]

recall@k calculation started
micro curves calculation started
fused rankings preparation started


In [19]:
plot_metrics_vs_window_with_stats(result_df, result_df)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAA=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [1],
               'y': [0.0]},
              {'line': {'color': 'purple', 'dash': 'dot'},
               'mode': 'lin

In [13]:
# generate function runs model for all dataset's elements and generates 3 files that are need for retrievel
index_path3 = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index3"
index3 = FaissFlatIndex.generate(
    directory=index_path3,
    dataset=three_rscan_ds,
    dataloader=None,
    model=model2,
    rebuild_meta=False,
    rebuild_descriptors=True,
    batch_size = 24,
    num_workers = 6,
    shuffle = False,
    metric = "l2", # can be also "ip" - inner product
    version = 1)
print(f"Index created at {index_path}")
print(f"Index size: {index.size()}, dim: {index.dim()} metric: {index.metric()}")

2026-05-05 12:58:30.101 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
100%|██████████| 394/394 [02:05<00:00,  3.15it/s]
2026-05-05 13:00:36.182 | INFO     | mmpr.inference.index:generate:465 - descriptors.npy file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index3
2026-05-05 13:00:36.786 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index3


Index created at /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/index
Index size: 9449, dim: 256 metric: l2


In [14]:
megaloc_pipeline = PlaceRecognitionPipeline(
    index=index3,
    model=model2,
    device="cuda",
)

In [15]:
query_cache_path = test_dir / "query_cache_megaloc"
megaloc_inferencer = PRInferencer(
    pr_pipeline=megaloc_pipeline,
    query_dataset=three_rscan_q,
    batch_size=16,
    num_workers=4,
    query_cache_dir=query_cache_path,
    k=100,
    device="cuda"
)

In [16]:
frames_megaloc = megaloc_inferencer.run(rebuild_query_descriptors=True)
megaloc_inferencer.save(query_cache_path / "frames_megaloc.npz", frames=frames_megaloc)

Compute descriptors + PR cache:  30%|███       | 399/1314 [02:26<05:26,  2.80it/s]2026-05-05 13:04:28.819 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:385 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000017.pt
2026-05-05 13:04:28.838 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:385 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000018.pt
Compute descriptors + PR cache: 100%|██████████| 1314/1314 [07:59<00:00,  2.74it/s]
2026-05-05 13:10:03.103 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-03/makarov-graphs/3rscan/query_cache_megaloc/meta.parquet


In [18]:
megaloc_inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:03<00:00, 6968.28it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,18586,0.884500,88.450007
1,5,21013,19632,0.934279,93.427878
2,10,21013,20010,0.952268,95.226764
3,25,21013,20434,0.972446,97.244563


In [19]:
bench_report_dir = Path("/home/kartashov_ga/projects/GSLoc/data/tests/26-04-25/Megaloc/Score-func/3rscan") / "seq_benchmark_report"
curr_report_dir = bench_report_dir / "room_k25_batch-std_megaloc"

megaloc_result_df = megaloc_inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    per_frame_k_used=25,
    final_k=25,
    save_dir=curr_report_dir,
    std_mode="global",
    scene_df_field="scene",
)

  0%|          | 0/11 [00:00<?, ?it/s]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


  9%|▉         | 1/11 [00:25<04:19, 25.92s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 18%|█▊        | 2/11 [00:51<03:53, 25.97s/it]

...

recall@k calculation started
micro curves calculation started
fused rankings preparation started
rankings creation started
ranking iteration started


 91%|█████████ | 10/11 [04:23<00:26, 26.69s/it]

recall@k calculation started
micro curves calculation started
fused rankings preparation started
rankings creation started
ranking iteration started


100%|██████████| 11/11 [04:50<00:00, 26.44s/it]

recall@k calculation started
micro curves calculation started
fused rankings preparation started


In [21]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

def plot_metrics_vs_window_with_stats(
    summary_df,
    summary_all,
    metrics = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Plot per-map metrics vs w and overlay cross-map mean and weighted mean.

    Args:
        summary_df: DataFrame for a single map (has columns 'w', metrics, and optional '<metric>_std').
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    df = summary_df.sort_values("w").reset_index(drop=True)
    map_name = "all"

    for m in metrics:
        if m not in df.columns:
            continue

        std_col = f"{m}_std"
        has_std = std_col in df.columns

        line_kwargs = {
            "x": "w",
            "y": m,
            "title": f"{map_name}: {m} vs sequence length (w)",
            "markers": True,
        }
        if has_std:
            line_kwargs["error_y"] = std_col

        fig = px.line(df, **line_kwargs)
        fig.update_layout(xaxis_title="sequence length (max_window)", yaxis_title=m)

        # Highlight maximum point on per-map line
        try:
            idx_max = df[m].astype(float).idxmax()
            w_star = int(df.loc[idx_max, "w"])  # sequence length at max
            y_star = float(df.loc[idx_max, m])
            fig.add_trace(
                go.Scatter(x=[w_star], y=[y_star], mode="markers", marker=dict(color="red", size=10), name="max", showlegend=False)
            )
            try:
                fig.add_vline(x=w_star, line_dash="dash", line_color="red")
            except Exception:
                fig.add_shape(type="line", x0=w_star, x1=w_star, y0=min(df[m].astype(float)), y1=max(df[m].astype(float)), line=dict(color="red", dash="dash"))
            fig.add_annotation(x=w_star, y=y_star, text=f"w={w_star}, {m}={y_star:.4f}", showarrow=True, arrowhead=2, ax=40, ay=-40)
        except Exception:
            pass

        # Overlay weighted mean across maps (weights = num_valid per map)
        try:
            wmean_series = (
                summary_all
                .groupby("w")
                .apply(lambda g: float(np.average(g[m].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                .reset_index(name=m)
            )

            weighted_mean_kwargs = {
                "x": wmean_series["w"],
                "y": wmean_series[m].astype(float),
                "mode": "lines",
                "name": "weighted mean",
                "line": dict(color="purple", dash="dot"),
                "showlegend": True,
            }

            if std_col in summary_all.columns:
                wstd_series = (
                    summary_all
                    .groupby("w")
                    .apply(lambda g: float(np.average(g[std_col].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                    .reset_index(name=std_col)
                )
                wmean_series = wmean_series.merge(wstd_series, on="w", how="left")
                weighted_mean_kwargs["error_y"] = dict(type="data", array=wmean_series[std_col].astype(float), visible=True)

            fig.add_trace(go.Scatter(**weighted_mean_kwargs))
        except Exception:
            pass

        figs[m] = fig
        fig.show()
    return figs


In [ ]:
plot_metrics_vs_window_with_stats(megaloc_result_df, megaloc_result_df)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAA=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [1],
               'y': [0.0]},
              {'line': {'color': 'purple', 'dash': 'dot'},
               'mode': 'lin

: 